# Now to actually make map

In [1]:
import os
import json

import numpy as np
import pandas as pd

from ipywidgets import Dropdown

from bqplot import Lines, Figure, LinearScale, DateScale, Axis

from ipyleaflet import Map, GeoJSON, WidgetControl, basemaps

import plotly.graph_objects as go

In [2]:
%run graph_programs/custom_colors.py
%run graph_programs/female_percentage.py


# Exploratory Data Analysis

In [3]:
# data = pd.read_json(os.path.abspath("nations.json"))

# converting shpae file to geojson
import geopandas as gpd

geo_json_path = os.path.join("geos", "cc20_us_03102025_500k_small.geojson")
gdf_small = gpd.read_file(geo_json_path)  # Load the existing GeoJSON file
    

In [4]:
print(gdf_small.columns)
geoids = gdf_small["GEOID"].astype(str).tolist()
print(geoids) 

bounds = gdf_small.total_bounds

Index(['CCN20', 'GEOID', 'STCD', 'STFIPS', 'STAB', 'D001TPOP', 'D0020004',
       'D0030509', 'D0041014', 'D0051519',
       ...
       'D154VSEAS', 'D155VOTHR', 'D156HVACR', 'D157RVACR', 'D158OCCHU',
       'D159OCCHUO', 'D160OCCHUR', 'DIVERSITY', 'DEPRATIO', 'geometry'],
      dtype='str', length=168)
['0101001', '0101002', '0101003', '0101004', '0101005', '0101006', '0101007', '0101008', '0101009', '0101010', '0101011', '0101012', '0101013', '0101014', '0101015', '0101016', '0102001', '0102002', '0102003', '0102004', '0102005', '0102006', '0102007', '0102008', '0102009', '0102010', '0102011', '0102012', '0102013', '0102014', '0102015', '0102016', '0102017', '0102018', '0103001', '0103002', '0103003', '0103004', '0103005', '0103006', '0103007', '0103008', '0103009', '0103010', '0103011', '0103012', '0103013', '0103014', '0103015', '0103016', '0103017', '0104001', '0104002', '0104003', '0104004', '0104005', '0104006', '0104007', '0104008', '0104009', '0104010', '0104011', '0104012', '

In [5]:
var_total_pop = ["D001TPOP"]
var_gender_pop = ["D025MALE", "D049FEMALE"]
var_age_brackets = ['D0020004', "D0030509", "D0041014", "D0051519","D0062024",
                    "D0072529","D0083034", "D0093539", "D0104044", "D0114549", 
                    "D0125054", "D0135559", "D0146064", "D0156569", "D0167074", 
                    "D0177579","D0188084", 'D01985UP']

demographic_var = var_total_pop + var_gender_pop + var_age_brackets
print(demographic_var)

['D001TPOP', 'D025MALE', 'D049FEMALE', 'D0020004', 'D0030509', 'D0041014', 'D0051519', 'D0062024', 'D0072529', 'D0083034', 'D0093539', 'D0104044', 'D0114549', 'D0125054', 'D0135559', 'D0146064', 'D0156569', 'D0167074', 'D0177579', 'D0188084', 'D01985UP']


In [6]:
print(gdf_small.head())

     CCN20    GEOID  STCD STFIPS STAB  D001TPOP  D0020004  D0030509  D0041014  \
0  0101001  0101001  0101     01   AL     55599      3155      3250      3638   
1  0101002  0101002  0101     01   AL     67916      4254      4607      5022   
2  0101003  0101003  0101     01   AL     34347      1947      2148      2355   
3  0101004  0101004  0101     01   AL     45525      2485      2895      3271   
4  0101005  0101005  0101     01   AL     36757      2183      2206      2416   

   D0051519  ...  D154VSEAS  D155VOTHR  D156HVACR  D157RVACR  D158OCCHU  \
0      3515  ...       1241       1319       0.25       0.16      21088   
1      4759  ...         62        573       0.14       0.05      25038   
2      2380  ...        105        520       0.17       0.08      13267   
3      3310  ...        681        620       0.21       0.12      17129   
4      2195  ...        220       1217       0.22       0.14      14499   

   D159OCCHUO  D160OCCHUR  DIVERSITY  DEPRATIO  \
0       1561

In [7]:
# cleaning data for proof
data = gdf_small[['GEOID'] + demographic_var][gdf_small["GEOID"].astype(str).isin(geoids)]
print(data.head())

     GEOID  D001TPOP  D025MALE  D049FEMALE  D0020004  D0030509  D0041014  \
0  0101001     55599     27908       27691      3155      3250      3638   
1  0101002     67916     32845       35071      4254      4607      5022   
2  0101003     34347     16594       17753      1947      2148      2355   
3  0101004     45525     22560       22965      2485      2895      3271   
4  0101005     36757     18254       18503      2183      2206      2416   

   D0051519  D0062024  D0072529  ...  D0104044  D0114549  D0125054  D0135559  \
0      3515      3228      3269  ...      3292      3300      3760      4209   
1      4759      3901      4028  ...      4363      4553      4376      4637   
2      2380      2080      2063  ...      2056      2213      2146      2358   
3      3310      2652      2419  ...      2942      3023      3019      3166   
4      2195      1935      2249  ...      2215      2504      2271      2554   

   D0146064  D0156569  D0167074  D0177579  D0188084  D01985UP 

# Making Base Map

In [8]:
m = Map(basemap=basemaps.CartoDB.Positron, 
        zoom=5)

hover_style = {'color': cc_red, 
               'fillColor': cc_red, 
               'opacity': 1.0, 
               'weight': 1.5, 
               'fillOpacity': 0.4}

# this makes the map tiles
geo = GeoJSON(
    data=gdf_small.__geo_interface__,
    style={"fillColor": "white", "weight": 0.5, 'color': cc_blue},
    hover_style= hover_style,
    name="GEOID",
)

m.add(geo)

# Fit map to bounds: [[south, west], [north, east]]
m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])

m

Map(center=[0.0, 0.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

# Adding Widgets

In [9]:
# Highlight overlay — starts empty, only ever holds the small selected subset
highlight_layer = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style={"fillColor": cc_black, "color": cc_black, "weight": 1.5, "fillOpacity": 0.4},
)

m.add_layer(highlight_layer)

In [10]:
fig_percent = make_female_percentage_graph()
fig_percent.show()

In [11]:
from bqplot import Figure, Label, LinearScale


# Fixed-range scales just for figure-relative positioning
label_scale_x = LinearScale(min=0, max=1)
label_scale_y = LinearScale(min=0, max=1)

hover_geoid = Label(
    text=['geoid'],
    x=[0.02], y=[0.75],
    scales={'x': label_scale_x, 'y': label_scale_y},
    font_size='20px',
    font_weight='bold',
    colors=cc_red,
)

hover_text = Label(
    text=['Total population'],
    x=[0.02], y=[0.45],
    scales={'x': label_scale_x, 'y': label_scale_y},
    font_size='20px',
    colors=cc_blue,
)

hover_pop_value = Label(
    text=['pop number'],
    x=[0.02], y=[0.3],
    scales={'x': label_scale_x, 'y': label_scale_y},
    font_size='20px',
    colors=cc_blue,
)

# Display the figure
hover_fig = Figure(
    marks=[hover_geoid, hover_text, hover_pop_value],
    layout={"width": "150px", "height": "110px"},
    fig_margin={"top": 0.01, "bottom": 0.01, "left": 0.01, "right": 0.01},
    min_aspect_ratio=0, max_aspect_ratio=100,
)


hover_fig

Figure(fig_margin={'top': 0.01, 'bottom': 0.01, 'left': 0.01, 'right': 0.01}, layout=Layout(height='110px', wi…

In [12]:
def update_hover_fig(geoid):
    # Filter the data for the selected GEOID
    filtered_data = data[data['GEOID'] == geoid]
    
    # Get the value of the demographic variable in this case, total pop
    value = filtered_data['D001TPOP'].values[0] if not filtered_data.empty else 'N/A'
    
    # Update the label text
    hover_geoid.text = [f"{geoid}"]
    hover_pop_value.text = [f"{value:,}"]



In [13]:
update_hover_fig('0101002')  # Example GEOID, replace with actual GEOID as needed
hover_fig

Figure(fig_margin={'top': 0.01, 'bottom': 0.01, 'left': 0.01, 'right': 0.01}, layout=Layout(height='110px', wi…

In [14]:
# adding widget on hover
widget_overview = WidgetControl(widget=hover_fig, position="bottomright")

m.add(widget_overview)

def on_hover(event, feature, **kwargs):
    global geoid
    geoid = feature["properties"]["GEOID"]
    update_hover_fig(geoid)  # Update the figure with the selected GEOID and demographic variable

geo.on_hover(on_hover)

m

Map(center=[np.float64(32.615677443261795), np.float64(-86.68073621090903)], controls=(ZoomControl(options=['p…

In [15]:
from ipywidgets import Button, Layout
import ipyleaflet
# I need a reset selectin button 
def reset_selection(button):
    selected_geoids.clear()  # Clear the selection
    update_highlight_layer([])  # Clear the highlight layer
    reset_female_percentage_graph(fig_percent)  # Reset
    update_right_panel(fig_percent)  # Update the right panel with the reset graph
    
reset_btn = Button(
    description='Reset Selection', 
    button_style='info',
    layout=Layout(width='auto', height='auto')
)

# changing button color
reset_btn.style.button_color = cc_blue

# adding reset button cabailities 
reset_btn.on_click(reset_selection)

widget_control = ipyleaflet.WidgetControl(widget=reset_btn, position='topright')
m.add_control(widget_control)

m

Map(center=[np.float64(32.615677443261795), np.float64(-86.68073621090903)], controls=(ZoomControl(options=['p…

In [16]:
from ipywidgets import HBox
import plotly.graph_objects as go

# Convert the static Figure into a live widget so it can sit in an HBox
right_panel = go.FigureWidget(fig_percent)
right_panel.layout.width = 350
right_panel.layout.height = 225

m_layout = HBox([m, right_panel])
m_layout

In [17]:
def update_right_panel_2(fig_percent):
    global m_layout
    
    # Convert the static Figure into a live widget so it can sit in an HBox
    right_panel = go.FigureWidget(fig_percent)
    right_panel.layout.width = 350
    right_panel.layout.height = 225

    m_layout = HBox([m, right_panel])
    
def update_right_panel(fig_percent):
    global right_panel
    right_panel.data = []
    for trace in fig_percent.data:
        right_panel.add_trace(trace)
    right_panel.layout.update(fig_percent.layout.to_plotly_json())

on click events
https://medium.com/swlh/how-to-use-mouse-events-on-ipyleaflet-4d002097efc0

In [18]:
def update_fig_side(geoids):
    # geoids can be a single value or a list — normalize to a list
    if not isinstance(geoids, (list, set, tuple)):
        geoids = [geoids]

    filtered_data = data[data['GEOID'].isin(geoids)]

    if filtered_data.empty:
        value = 'N/A'
    else:
        value = filtered_data[demographic_var].values  # array of values, one per selected region
        # decide here how to combine them if multiple selected, e.g.:
        # value = filtered_data[demographic_var].mean()  # average across selection
        # or keep as a list to show all selected values in the label/chart

    # Update the label text
    label_side.text = [str(value)]

In [19]:
def update_highlight_layer(geoids):
    # geoids can be a single value or a list — normalize to a list
    if not isinstance(geoids, (list, set, tuple)):
        geoids = [geoids]

    # Filter the GeoDataFrame for the selected GEOIDs
    filtered_gdf = gdf_small[gdf_small['GEOID'].astype(str).isin(geoids)]

    # Update the highlight layer with the new GeoJSON data
    highlight_layer.data = filtered_gdf.__geo_interface__

# Adding in gender decomp

In [20]:
print(data.head())

     GEOID  D001TPOP  D025MALE  D049FEMALE  D0020004  D0030509  D0041014  \
0  0101001     55599     27908       27691      3155      3250      3638   
1  0101002     67916     32845       35071      4254      4607      5022   
2  0101003     34347     16594       17753      1947      2148      2355   
3  0101004     45525     22560       22965      2485      2895      3271   
4  0101005     36757     18254       18503      2183      2206      2416   

   D0051519  D0062024  D0072529  ...  D0104044  D0114549  D0125054  D0135559  \
0      3515      3228      3269  ...      3292      3300      3760      4209   
1      4759      3901      4028  ...      4363      4553      4376      4637   
2      2380      2080      2063  ...      2056      2213      2146      2358   
3      3310      2652      2419  ...      2942      3023      3019      3166   
4      2195      1935      2249  ...      2215      2504      2271      2554   

   D0146064  D0156569  D0167074  D0177579  D0188084  D01985UP 

In [21]:
data["gender_female"] = round((data["D049FEMALE"] / data["D001TPOP"]) * 100, 2)
print(data[["GEOID", "gender_female"]].head())

     GEOID  gender_female
0  0101001          49.80
1  0101002          51.64
2  0101003          51.69
3  0101004          50.44
4  0101005          50.34


Now I want to incporate this into the graph

In [22]:
# initilaizing 
from graph_programs.female_percentage import update_female_percentage_graph

selected_geoids = set()

def handle_click(event=None, feature=None, **kwargs):
    global fig_percent
    geoid = feature["properties"]["GEOID"]

    if geoid in selected_geoids:
        selected_geoids.discard(geoid)
    else:
        selected_geoids.add(geoid)

    filtered = gdf_small[gdf_small['GEOID'].astype(str).isin(selected_geoids)]
    fig_percent = update_female_percentage_graph(
        demographics_data = data, 
        geoids_df = filtered, 
        fig_percent = fig_percent)
    update_right_panel(fig_percent)
    update_highlight_layer(list(selected_geoids))

geo.on_click(handle_click)

display(m_layout)


# Adding in Line Graph for Age Buckets
+ basically want comparaitve hline ohover and percentages 

In [23]:
print(data)

      GEOID  D001TPOP  D025MALE  D049FEMALE  D0020004  D0030509  D0041014  \
0   0101001     55599     27908       27691      3155      3250      3638   
1   0101002     67916     32845       35071      4254      4607      5022   
2   0101003     34347     16594       17753      1947      2148      2355   
3   0101004     45525     22560       22965      2485      2895      3271   
4   0101005     36757     18254       18503      2183      2206      2416   
..      ...       ...       ...         ...       ...       ...       ...   
95  0106013     50575     24200       26375      2817      3264      3459   
96  0106014     59881     29001       30880      2961      3233      3822   
97  0106015     40412     21570       18842      2257      2477      2588   
98  0106016     58805     28390       30415      3513      3796      4337   
99  0107001     40805     19551       21254      2271      2167      2401   

    D0051519  D0062024  D0072529  ...  D0114549  D0125054  D0135559  D01460

+ Source - https://stackoverflow.com/a/77169246
+ Posted by Timeless
+ Retrieved 2026-07-16, License - CC BY-SA 4.0

In [27]:
# making smaller age brackets 
data_new = data.assign(
      age_under15 = lambda x: x.pop('D0020004') + x.pop('D0030509') + x.pop('D0041014')
    ).assign(
      age_15_24 = lambda x: x.pop('D0051519')+ x.pop('D0062024')
   ).assign(
      age_25_34 = lambda x: x.pop('D0072529') + x.pop('D0083034')
   ).assign(
      age_35_44 = lambda x: x.pop('D0093539') + x.pop('D0104044')
   ).assign(
    age_45_54 = lambda x: x.pop('D0114549') + x.pop('D0125054')
   ).assign(
    age_55_64 = lambda x: x.pop('D0135559') + x.pop('D0146064')
   ).assign(
    age_over65 = lambda x: x.pop('D0156569') + x.pop('D0167074') + x.pop('D0177579') +
         x.pop('D0188084') + x.pop('D01985UP'))
   
(data_new)


,GEOID,D001TPOP,D025MALE,D049FEMALE,gender_female,age_under15,age_15_24,age_25_34,age_35_44,age_45_54,age_55_64,age_over65
0,0101001,55599,27908,27691,49.80,10043,6743,6569,6633,7060,8635,9916
1,0101002,67916,32845,35071,51.64,13883,8660,8491,8973,8929,8892,10088
2,0101003,34347,16594,17753,51.69,6450,4460,3944,4201,4359,4662,6271
3,0101004,45525,22560,22965,50.44,8651,5962,4881,5787,6042,6135,8067
4,0101005,36757,18254,18503,50.34,6805,4130,4477,4358,4775,5090,7122
...,...,...,...,...,...,...,...,...,...,...,...,...
95,0106013,50575,24200,26375,52.15,9540,5889,5956,6380,6434,7027,9349
96,0106014,59881,29001,30880,51.57,10016,6647,6359,7237,8148,9413,12061
97,0106015,40412,21570,18842,46.62,7322,5111,6084,5582,5661,5213,5439
98,0106016,58805,28390,30415,51.72,11646,7415,7282,7708,7740,7771,9243


In [ ]:
# extracting age brackets
age_brackets = data_new.filter(like="age").columns.tolist()

['age_under15', 'age_15_24', 'age_25_34', 'age_35_44', 'age_45_54', 'age_55_64', 'age_over65']


In [36]:
geoids_df = gdf_small
demographics_data = data_new

In [ ]:
# ❌ INEFFICIENT (O(N*M) loop-in-loop)
# for item_a in list_a:
#     for item_b in list_b:
#         if item_a == item_b:
#             match(item_a)

#   EFFICIENT (O(N) time, minor one-time memory footprint for the set)
set_b = set(list_b) 
matching_generator = (item for item in list_a if item in set_b)

In [42]:
age_brackets[0]

'age_under15'

In [ ]:
# building out line graph 

import plotly.graph_objects as go
from graph_programs.custom_colors import custom_continuous

fig_age = go.Figure()

# making backgroufn transparent
fig_age.update_layout({'plot_bgcolor': 'rgba(0, 0, 0, 0)', 'paper_bgcolor': 'rgba(0, 0, 0, 0)'})
    
# getting rid of grid lines
fig_age.update_xaxes(showgrid=False)
fig_age.update_yaxes(showgrid=False)

# indexes the data at GEOID
demo_indexed = demographics_data.set_index("GEOID")
# filters in specific GEOIDS 
subset = demo_indexed.loc[geoids_df["GEOID"], age_brackets]
# builds out dictiornary 
plt_data = {"GEOID": subset.index.tolist(), **{age: subset[age].tolist() for age in age_brackets}}



fig_age.show()


{'GEOID': ['0101001', '0101002', '0101003', '0101004', '0101005', '0101006', '0101007', '0101008', '0101009', '0101010', '0101011', '0101012', '0101013', '0101014', '0101015', '0101016', '0102001', '0102002', '0102003', '0102004', '0102005', '0102006', '0102007', '0102008', '0102009', '0102010', '0102011', '0102012', '0102013', '0102014', '0102015', '0102016', '0102017', '0102018', '0103001', '0103002', '0103003', '0103004', '0103005', '0103006', '0103007', '0103008', '0103009', '0103010', '0103011', '0103012', '0103013', '0103014', '0103015', '0103016', '0103017', '0104001', '0104002', '0104003', '0104004', '0104005', '0104006', '0104007', '0104008', '0104009', '0104010', '0104011', '0104012', '0104013', '0104014', '0104015', '0104016', '0104017', '0105001', '0105002', '0105003', '0105004', '0105005', '0105006', '0105007', '0105008', '0105009', '0105010', '0105011', '0105012', '0105013', '0105014', '0105015', '0106001', '0106002', '0106003', '0106004', '0106005', '0106006', '0106007',

In [41]:
print(list(plt_data.keys())[0])

age_under15


# Saving the map

In [25]:
from ipywidgets.embed import embed_minimal_html

# embed_minimal_html('./output/map_draft.html', views=[m_layout])